### 보드라이프 리뷰 데이터 전처리 보고서

---

**데이터 소스**
- 입력 파일: `boardlife_reviews_all.csv`
- 출력 파일: `boardlife_reviews_final.csv`
- 원본: 153,351행 / 8컬럼

---

**전처리 항목**

**1. 불필요 컬럼 제거**

8개 중 4개만 유지, 나머지 제거

| 제거 컬럼 | 제거 이유 |
| --- | --- |
| `game_id` | rank + title로 게임 특정 가능, 중복 |
| `reviewer` | 익명/비식별 데이터로 활용 가치 낮음 |
| `date` | `'4시간 전'`, `'2일 전'` 형태의 상대 시간으로 정규화 불가 |
| `review_page` | 크롤링 내부 번호로 분석에 불필요 |

유지한 컬럼:

| 컬럼 | 이유 |
| --- | --- |
| `rank` | 게임 식별 및 games 조인 키 |
| `title` | 게임 식별 및 games 조인 키 |
| `rating` | 평점 기반 필터링 및 stats 연계 |
| `review_text` | RAG 핵심 소스 |

**2. 텍스트 없는 행 제거**

| 처리 항목 | 처리 방식 | 해당 행 수 |
| --- | --- | --- |
| `review_text` null | 행 삭제 | 103,227행 (67.3%) |

평점만 기록하고 리뷰 텍스트를 남기지 않은 경우로, RAG 임베딩에 활용할 수 없어 삭제한다.

---

**games 연계**
- `rank + title` 기준으로 `boardlife_games_final.csv` 조인
- 리뷰 있는 게임 수: 3,238개 / 3,322개 (97.5%)
- 리뷰 없는 게임 수: 84개

---

**다음 단계 연계**
- `review_text` → ko-sroberta-multitask 임베딩 → FAISS 인덱싱
- `rating` → RAG 필터링 및 평점 기반 추천 보정에 활용

In [1]:
import pandas as pd

reviews = pd.read_csv('data/boardlife_reviews_all.csv')

print('shape:', reviews.shape)
print('columns:', reviews.columns.tolist())
print()
print('null 현황:')
print(reviews.isnull().sum())

shape: (153351, 8)
columns: ['rank', 'title', 'game_id', 'rating', 'reviewer', 'date', 'review_text', 'review_page']

null 현황:
rank                0
title               0
game_id             0
rating              0
reviewer         2453
date                0
review_text    103227
review_page         0
dtype: int64


---
## 1. 불필요 컬럼 제거

In [2]:
reviews = reviews.drop(columns=['game_id', 'reviewer', 'date', 'review_page'])
print('컬럼 제거 후:', reviews.columns.tolist())

컬럼 제거 후: ['rank', 'title', 'rating', 'review_text']


---
## 2. review_text null 행 제거

In [3]:
before = len(reviews)
reviews = reviews.dropna(subset=['review_text'])
after = len(reviews)

print(f'삭제된 행: {before - after:,}개 ({(before - after) / before * 100:.1f}%)')
print(f'남은 행: {after:,}개')
print()
print('null 확인:')
print(reviews.isnull().sum())

삭제된 행: 103,227개 (67.3%)
남은 행: 50,124개

null 확인:
rank           0
title          0
rating         0
review_text    0
dtype: int64


---
## 3. 최종 확인

In [4]:
print('shape:', reviews.shape)
print('columns:', reviews.columns.tolist())
print()
print('dtypes:')
print(reviews.dtypes)

shape: (50124, 4)
columns: ['rank', 'title', 'rating', 'review_text']

dtypes:
rank             int64
title              str
rating         float64
review_text        str
dtype: object


In [5]:
reviews.head(3)

,rank,title,rating,review_text
0,1,브라스: 버밍엄,9.0,갓겜 오브 갓겜이죠. 처음 룰만 잘 배워두면 두고두고 즐길 수 있습니다
1,1,브라스: 버밍엄,3.0,비직관적인 잔룰이 많고 운영도 누구한테 배우지 않으면 접근하기 어려움. 예를 들어 ...
3,1,브라스: 버밍엄,9.0,"웨이트에 비해 간결하고 잔룰이 없으며, 적절한 인터랙션이 있는 웰메이드 보드 게임...."


---
## 4. 저장

In [6]:
reviews.to_csv('data/boardlife_reviews_final.csv', index=False, encoding='utf-8-sig')
print(f'저장 완료: boardlife_reviews_final.csv → {reviews.shape[0]:,}행 {reviews.shape[1]}컬럼')

저장 완료: boardlife_reviews_final.csv → 50,124행 4컬럼
